# Fleet Maintenance Planning

**Team:** 38

### 1. Project Overview

#### Objective

Predict which vehicles should be inspected and determine
the appropriate spare-parts budget for each inspected vehicle.

The objective is to minimise the total maintenance cost
using the competition's specified cost function.

#### Project Workflow

1. Data loading and exploration
2. Preprocessing and feature selection
3. Failure prediction
4. Repair-cost prediction
5. Cost-based decision optimisation
6. Final submission generation

### 2. Imports and Config

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the project root without changing the working directory.
current = Path.cwd().resolve()

ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data").is_dir() and (p / "src").is_dir()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Could not locate project root containing data/ and src/."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Import all shared paths and settings from one place.
from src.config import (
    DATA_DIR,
    RESULTS_DIR,
    SUBMISSIONS_DIR,
    RANDOM_STATE,
)

print("Project root:", ROOT)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)

Project root: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38
Data directory: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38\data
Results directory: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38\results


In [15]:
import numpy as np
from src.evaluation import (
    get_repair_cost_ratios,
    predict_mixture_budgets,
    tune_v3_policy,
    apply_v3_policy,
)

# Synthetic example
actual = np.array([100, 200, 300, 400, 500], dtype=float)
predicted = np.array([110, 180, 280, 420, 480], dtype=float)
probabilities = np.array([0.05, 0.10, 0.30, 0.70, 0.95])

ratios = get_repair_cost_ratios(actual, predicted)

budgets = predict_mixture_budgets(
    probabilities, predicted, ratios
)

policy = tune_v3_policy(
    probabilities, actual, predicted, ratios
)

flags, final_budgets = apply_v3_policy(
    probabilities, predicted, ratios, policy["threshold"]
)

print("Ratios:", ratios)
print("Mixture budgets:", budgets)
print("Selected policy:", policy)
print("Inspection flags:", flags)
print("Final budgets:", final_budgets)

assert np.isfinite(final_budgets).all()
assert (final_budgets >= 0).all()
assert budgets[0] == 0
assert np.array_equal(flags, (probabilities >= policy["threshold"]).astype(int))

print("V3 policy sanity test passed.")

Ratios: [0.90909091 0.95238095 1.04166667 1.07142857 1.11111111]
Mixture budgets: [  0.         166.46989374 298.23232323 458.00865801 526.04237867]
Selected policy: {'cost': 78.74057733866334, 'threshold': 0.05}
Inspection flags: [1 1 1 1 1]
Final budgets: [  0.         166.46989374 298.23232323 458.00865801 526.04237867]
V3 policy sanity test passed.


### 3. Data Loading:

In [2]:
from src.feature_analysis import load_data

# Load training and test datasets
train_df, test_df = load_data(DATA_DIR)

# Load sample submission for final format validation
sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

print("Training data:", train_df.shape)
print("Testing data:", test_df.shape)
print("Sample submission:", sample_submission.shape)

Training data: (56000, 174)
Testing data: (20000, 171)
Sample submission: (20000, 3)


### 4. Data Validation

In [3]:
# Check required columns
assert "failure" in train_df.columns
assert "repair_cost" in train_df.columns
assert "id" in train_df.columns

# Check identifiers
assert train_df["id"].is_unique
assert test_df["id"].is_unique

# Check target
assert train_df["failure"].notna().all()
assert train_df["failure"].isin([0, 1]).all()

# Check feature compatibility
sensor_cols = [
    col for col in train_df.columns
    if col.startswith("s")
]

assert set(sensor_cols) == {
    col for col in test_df.columns
    if col.startswith("s")
}

print("Basic data validation passed!")

Basic data validation passed!


### 5. EDA Summary

- The failure rate is highly imbalanced (0.71%).
- Repair costs are strongly right-skewed.
- Sensor features contain substantial missing values.
- Eight sensors have more than 50% missing values.
- One sensor (`s146`) is constant.
- Several sensors show substantial train–test differences, suggesting potential distribution shift between depots.

### 6. Cross-Validation Setup

In [4]:
from src.cv_utils import load_folds, validate_folds
from src.lightgbm_model import get_base_features

# Select features available and usable at test time
features = get_base_features(train_df, test_df)

X = train_df[features]
y = train_df["failure"]

# Load and validate the shared CV folds
folds = load_folds()
validate_folds(folds, len(X))

print(f"Training vehicles: {len(X):,}")
print(f"Features selected: {len(features)}")
print(f"CV folds: {len(folds)}")
print(f"Features excluded: {train_df.shape[1] - len(features) - 3}")

Fold validation successful.
Training vehicles: 56,000
Features selected: 167
CV folds: 5
Features excluded: 4


In [5]:
from src.repair_model import generate_oof_log_repair_costs

oof_log_budgets = generate_oof_log_repair_costs(
    train_df=train_df,
    features=features,
    folds=folds,
)

print("Shape:", oof_log_budgets.shape)
print("All finite:", np.isfinite(oof_log_budgets).all())
print("Minimum:", oof_log_budgets.min())
print("Maximum:", oof_log_budgets.max())
print("Mean:", oof_log_budgets.mean())

V3 log repair-cost OOF fold 1/5
V3 log repair-cost OOF fold 2/5
V3 log repair-cost OOF fold 3/5
V3 log repair-cost OOF fold 4/5
V3 log repair-cost OOF fold 5/5
Shape: (56000,)
All finite: True
Minimum: 3.4944926258919082
Maximum: 2582.14107242053
Mean: 209.11082362818053


### 7. Out-of-Fold Predictions

In [17]:

# ==========================================
# 7. V3 OUT-OF-FOLD PREDICTIONS
# ==========================================

from src.ensemble_model import (
    FailureProbabilityEnsemble,
    family_oof_probabilities,
)
from src.lightgbm_model import run_cv
from src.repair_model import generate_oof_log_repair_costs

# 1. Generate failure-probability OOF predictions.
ensemble_cv = run_cv(
    model=FailureProbabilityEnsemble(),
    X=X,
    y=y,
    folds=folds,
    name="V3 Failure Ensemble",
    verbose=True,
)

oof_v3_probabilities = family_oof_probabilities(
    cv_result=ensemble_cv,
    X=X,
    folds=folds,
)

# 2. Generate log-transformed repair-cost OOF predictions.
oof_v3_repair_costs = generate_oof_log_repair_costs(
    train_df=train_df,
    features=features,
    folds=folds,
)

# 3. Validate predictions.
assert len(oof_v3_probabilities) == len(train_df)
assert len(oof_v3_repair_costs) == len(train_df)

assert np.isfinite(oof_v3_probabilities.to_numpy()).all()
assert np.isfinite(oof_v3_repair_costs).all()

assert (
    (oof_v3_probabilities >= 0)
    & (oof_v3_probabilities <= 1)
).all().all()

assert (oof_v3_repair_costs >= 0).all()

print("\nV3 OOF generation complete.")
print("Classifier predictions:", oof_v3_probabilities.shape)
print("Repair-cost predictions:", oof_v3_repair_costs.shape)

display(oof_v3_probabilities.describe())
print("\nRepair-cost prediction summary:")
print(pd.Series(oof_v3_repair_costs).describe())


Fold validation successful.
V3 Failure Ensemble | fold 1: ROC-AUC 0.9841 | PR-AUC 0.6940 | 102.6s
V3 Failure Ensemble | fold 2: ROC-AUC 0.9837 | PR-AUC 0.5968 | 102.3s
V3 Failure Ensemble | fold 3: ROC-AUC 0.9796 | PR-AUC 0.5065 | 103.5s
V3 Failure Ensemble | fold 4: ROC-AUC 0.9922 | PR-AUC 0.4588 | 103.6s
V3 Failure Ensemble | fold 5: ROC-AUC 0.9823 | PR-AUC 0.5696 | 104.3s
V3 log repair-cost OOF fold 1/5
V3 log repair-cost OOF fold 2/5
V3 log repair-cost OOF fold 3/5
V3 log repair-cost OOF fold 4/5
V3 log repair-cost OOF fold 5/5

V3 OOF generation complete.
Classifier predictions: (56000, 4)
Repair-cost predictions: (56000,)


,catboost,xgboost,hist_gradient_boosting,ensemble
count,56000.000000,56000.000000,5.600000e+04,56000.000000
mean,0.005793,0.005688,4.737529e-03,0.005406
std,0.056491,0.055190,5.734716e-02,0.054913
min,0.000004,0.000002,1.305423e-07,0.000003
25%,0.000033,0.000025,1.080593e-06,0.000024
50%,0.000054,0.000051,1.980998e-06,0.000039
75%,0.000105,0.000107,4.790694e-06,0.000077
max,0.994083,0.997175,9.996094e-01,0.993605



Repair-cost prediction summary:
count    56000.000000
mean       209.110824
std        155.818706
min          3.494493
25%        123.825744
50%        160.938981
75%        234.076472
max       2582.141072
dtype: float64


In [19]:

# ==========================================
# 8. V3 POLICY OPTIMISATION
# ==========================================

from src.evaluation import (
    get_repair_cost_ratios,
    tune_v3_policy,
    apply_v3_policy,
    calculate_cost,
)

actual_costs = train_df["repair_cost"].fillna(0).to_numpy()
predicted_costs = np.asarray(oof_v3_repair_costs)

# Estimate repair-cost ratios using OOF predictions on actual failures.
repair_cost_ratios = get_repair_cost_ratios(
    actual_costs=train_df["repair_cost"].to_numpy(),
    predicted_budgets=predicted_costs,
)

print("Repair-cost ratios:", len(repair_cost_ratios))
print("Ratio summary:", pd.Series(repair_cost_ratios).describe())

# Evaluate each classifier using the V3 policy.
v3_policy_results = []

for model_name in oof_v3_probabilities.columns:
    probabilities = oof_v3_probabilities[model_name].to_numpy()

    policy = tune_v3_policy(
        probabilities=probabilities,
        actual_costs=actual_costs,
        predicted_costs=predicted_costs,
        ratios=repair_cost_ratios,
    )

    flags, budgets = apply_v3_policy(
        probabilities=probabilities,
        predicted_costs=predicted_costs,
        ratios=repair_cost_ratios,
        threshold=policy["threshold"],
    )

    total_cost = calculate_cost(
        repair_costs=actual_costs,
        flags=flags,
        budgets=budgets,
    )
    
    v3_policy_results.append({
        "model": model_name,
        "total_cost": total_cost,
        "threshold": policy["threshold"],
        "vehicles_inspected": int(flags.sum()),
        "inspection_rate": flags.mean(),
    })

v3_policy_comparison = (
    pd.DataFrame(v3_policy_results)
    .sort_values("total_cost")
    .reset_index(drop=True)
)

display(v3_policy_comparison)


Repair-cost ratios: 397
Ratio summary: count    397.000000
mean       1.542368
std        2.025236
min        0.050557
25%        0.521291
50%        0.969551
75%        1.745653
max       18.494536
dtype: float64


,model,total_cost,threshold,vehicles_inspected,inspection_rate
0,ensemble,50619.062412,0.006790,1583,0.028268
1,catboost,51510.967625,0.008085,1785,0.031875
2,xgboost,52538.643650,0.004534,2048,0.036571
3,hist_gradient_boosting,53856.417623,0.000736,1602,0.028607


### 8. Cost-Based Policy Optimisation

In [7]:
from src.evaluation import tune_policy, calculate_cost

# Find the best inspection threshold and repair-budget multiplier.
policy = tune_policy(
    probabilities=oof_failure_probabilities,
    actual_costs=train_df["repair_cost"],
    predicted_budgets=oof_repair_budgets,
)

best_threshold = policy["threshold"]
best_multiplier = policy["multiplier"]

# Calculate the resulting inspection decisions.
oof_flags = (
    oof_failure_probabilities >= best_threshold
).astype(int)

oof_budgets = np.where(
    oof_flags == 1,
    oof_repair_budgets * best_multiplier,
    0.0,
)

# Independently calculate the selected policy's official cost.
verified_cost = calculate_cost(
    repair_costs=train_df["repair_cost"],
    flags=oof_flags,
    budgets=oof_budgets,
)

print("Selected inspection threshold:", best_threshold)
print("Selected budget multiplier:", best_multiplier)
print(f"OOF policy cost: {verified_cost:,.2f}")
print(f"Vehicles inspected: {oof_flags.sum():,}")
print(f"Inspection rate: {oof_flags.mean():.2%}")

Selected inspection threshold: 0.000303746594408579
Selected budget multiplier: 0.75
OOF policy cost: 59,210.58
Vehicles inspected: 1,278
Inspection rate: 2.28%


### 9. Final Model Training and Test Predictions

In [21]:

# ==========================================
# 9. V3 FINAL MODEL TRAINING AND TEST PREDICTIONS
# ==========================================

from src.ensemble_model import FailureProbabilityEnsemble
from src.feature_analysis import align_test_sensors
from src.repair_model import fit_final_log_repair_model

# 1. Align test sensors with training features.
test_aligned = align_test_sensors(test_df, train_df)
X_test = test_aligned[features]

# 2. Train the V3 failure-probability ensemble.
final_failure_model = FailureProbabilityEnsemble()
final_failure_model.fit(X, y)

test_failure_probabilities = (
    final_failure_model.predict_failure_probability(X_test)
)

# 3. Train the V3 log repair-cost model.
final_repair_model, final_repair_smear, test_repair_costs = (
    fit_final_log_repair_model(
        train_df=train_df,
        test_df=test_aligned,
        features=features,
    )
)

# 4. Select the best OOF policy.
best_policy = v3_policy_comparison.iloc[0]
best_threshold = float(best_policy["threshold"])

# 5. Apply the V3 policy to test predictions.
from src.evaluation import apply_v3_policy

test_flags, test_budgets = apply_v3_policy(
    probabilities=test_failure_probabilities,
    predicted_costs=test_repair_costs,
    ratios=repair_cost_ratios,
    threshold=best_threshold,
)

# 6. Validate test predictions.
assert len(test_failure_probabilities) == len(test_df)
assert len(test_repair_costs) == len(test_df)
assert len(test_flags) == len(test_df)
assert len(test_budgets) == len(test_df)

assert np.isfinite(test_failure_probabilities).all()
assert np.isfinite(test_repair_costs).all()
assert np.isfinite(test_budgets).all()

assert ((test_failure_probabilities >= 0) &
        (test_failure_probabilities <= 1)).all()
assert (test_repair_costs >= 0).all()
assert (test_budgets >= 0).all()
assert np.isin(test_flags, [0, 1]).all()

print("V3 final models trained successfully.")
print("Selected classifier:", best_policy["model"])
print("Selected threshold:", best_threshold)
print("Test failure probabilities:", test_failure_probabilities.shape)
print("Test repair-cost predictions:", test_repair_costs.shape)
print("Vehicles selected for inspection:", test_flags.sum())
print("Inspection rate:", f"{test_flags.mean():.2%}")


V3 final models trained successfully.
Selected classifier: ensemble
Selected threshold: 0.0067904391341569005
Test failure probabilities: (20000,)
Test repair-cost predictions: (20000,)
Vehicles selected for inspection: 1126
Inspection rate: 5.63%


### 10. Submission Generation and Validation

In [23]:

# ==========================================
# 10. V3 SUBMISSION GENERATION AND VALIDATION
# ==========================================

# Construct submission using V3 predictions.
submission = pd.DataFrame({
    "id": test_df["id"].to_numpy(),
    "flag": test_flags,
    "budget": test_budgets,
})

# Validate submission.
assert len(submission) == len(test_df)
assert submission["id"].is_unique
assert submission["id"].equals(test_df["id"])
assert submission["flag"].isin([0, 1]).all()
assert np.isfinite(submission["budget"]).all()
assert (submission["budget"] >= 0).all()
assert submission.notna().all().all()

# Validate against sample submission.
assert list(submission.columns) == list(sample_submission.columns)
assert submission["id"].equals(sample_submission["id"])

# Save V3 submission separately.
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

output_path = SUBMISSIONS_DIR / "SUBMISSION_Ensemble_LogLightGBM.csv"
submission.to_csv(output_path, index=False)

print("V3 submission generated successfully!")
print("Output:", output_path)
print(f"Vehicles: {len(submission):,}")
print(f"Inspections: {test_flags.sum():,}")
print(f"Inspection rate: {test_flags.mean():.2%}")
print(f"Mean predicted failure probability: {test_failure_probabilities.mean():.6f}")

if test_flags.sum() > 0:
    print(
        "Mean budget among inspected:",
        f"{test_budgets[test_flags == 1].mean():.2f}"
    )

print("\nV3 submission validation passed.")


V3 submission generated successfully!
Output: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38\submissions\SUBMISSION_Ensemble_LogLightGBM.csv
Vehicles: 20,000
Inspections: 1,126
Inspection rate: 5.63%
Mean predicted failure probability: 0.016934
Mean budget among inspected: 377.36

V3 submission validation passed.


In [10]:
from src.classifier_experiments import run_classifier_experiments

experiment_results, experiment_oof = run_classifier_experiments(
    train_df=train_df,
    features=features,
    folds=folds,
    oof_repair_budgets=oof_repair_budgets,
    baseline_probabilities=oof_failure_probabilities,
)

display(experiment_results)



Running random_forest
Fold 1/5 | ROC-AUC: 0.9865 | PR-AUC: 0.5318 | Time: 3.9s
Fold 2/5 | ROC-AUC: 0.9845 | PR-AUC: 0.4748 | Time: 3.9s
Fold 3/5 | ROC-AUC: 0.9796 | PR-AUC: 0.4190 | Time: 3.7s
Fold 4/5 | ROC-AUC: 0.9879 | PR-AUC: 0.3791 | Time: 3.8s
Fold 5/5 | ROC-AUC: 0.9732 | PR-AUC: 0.4923 | Time: 3.6s
random_forest OOF generation complete.

Running xgboost
Fold 1/5 | ROC-AUC: 0.9901 | PR-AUC: 0.6637 | Time: 6.6s
Fold 2/5 | ROC-AUC: 0.9849 | PR-AUC: 0.5770 | Time: 6.7s
Fold 3/5 | ROC-AUC: 0.9777 | PR-AUC: 0.5170 | Time: 6.5s
Fold 4/5 | ROC-AUC: 0.9900 | PR-AUC: 0.4199 | Time: 6.7s
Fold 5/5 | ROC-AUC: 0.9881 | PR-AUC: 0.5451 | Time: 6.6s
xgboost OOF generation complete.

lightgbm: cost=59,210.58, ROC-AUC=0.9849, PR-AUC=0.5261

random_forest: cost=63,634.49, ROC-AUC=0.9823, PR-AUC=0.4468

xgboost: cost=59,626.46, ROC-AUC=0.9860, PR-AUC=0.5204


,model,roc_auc,pr_auc,oof_cost,threshold,budget_multiplier,inspections,inspection_rate
0,lightgbm,0.984860,0.526133,59210.577681,0.000304,0.75,1278,0.022821
2,xgboost,0.986002,0.520363,59626.462817,0.046665,0.75,1362,0.024321
1,random_forest,0.982298,0.446781,63634.491332,0.052642,0.75,1612,0.028786


In [11]:

# ==========================================
# XGBoost submission candidate
# ==========================================

from src.classifier_experiments import build_experiment_models
from src.preprocessing import fit_preprocessor, transform_features
from src.feature_analysis import align_test_sensors

# Selected from XGBoost OOF policy optimisation
xgb_threshold = 0.046665
xgb_multiplier = 0.75

# Align test sensors exactly as in the baseline pipeline
test_aligned = align_test_sensors(test_df, train_df)

X_train = train_df[features]
X_test = test_aligned[features]
y_train = train_df["failure"].to_numpy()

# Fit preprocessing on the complete training data
xgb_prep = fit_preprocessor(
    X_train,
    imputation="median",
    scale=False,
)

X_train_processed = transform_features(xgb_prep, X_train)
X_test_processed = transform_features(xgb_prep, X_test)

# Train XGBoost on all training data
xgb_final = build_experiment_models(y_train)["xgboost"]
xgb_final.fit(X_train_processed, y_train)

# Predict test failure probabilities
xgb_test_probabilities = xgb_final.predict_proba(
    X_test_processed
)[:, 1]

# Apply OOF-selected policy
xgb_test_flags = (
    xgb_test_probabilities >= xgb_threshold
).astype(int)

xgb_test_budgets = np.where(
    xgb_test_flags == 1,
    test_repair_budgets * xgb_multiplier,
    0.0,
)

# Construct submission
xgb_submission = pd.DataFrame({
    "id": test_df["id"].to_numpy(),
    "flag": xgb_test_flags,
    "budget": xgb_test_budgets,
})

# Validate
assert len(xgb_submission) == len(sample_submission)
assert xgb_submission["id"].equals(sample_submission["id"])
assert xgb_submission["flag"].isin([0, 1]).all()
assert xgb_submission["budget"].notna().all()
assert (xgb_submission["budget"] >= 0).all()

# Save separately; preserve the current submission
xgb_path = SUBMISSIONS_DIR / "SUBMISSION_XGBOOST.csv"
xgb_submission.to_csv(xgb_path, index=False)

print("XGBoost submission generated!")
print("File:", xgb_path)
print("Inspections:", xgb_test_flags.sum())
print("Inspection rate:", f"{xgb_test_flags.mean():.2%}")
print("Mean predicted probability:", xgb_test_probabilities.mean())
print("Mean inspected budget:", 
      xgb_test_budgets[xgb_test_flags == 1].mean())


XGBoost submission generated!
File: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38\submissions\SUBMISSION_XGBOOST.csv
Inspections: 1034
Inspection rate: 5.17%
Mean predicted probability: 0.0330224
Mean inspected budget: 393.3051291017363
